# Scene-sheet formats: a 40-question exploration

Qwen3.5-2B (thinking off, 16 frames, `uniform128`, up to 2,048 new tokens, final answer scored) on 40 VSI-Bench
questions from 5 scenes (`explore_ids.txt`): 5 per task, relative direction split 2 easy / 2 medium / 1 hard, and 3
questions whose pipeline sheet is empty while the gold sheet has the object. Eight conditions: **A** video only; the
pipeline and gold sheets as csv and as markdown_kv; and three made by `explore_sheets.py`, all csv:

- **`pipeline_oracle_scale`**, an **oracle upper bound** (not a usable method): the pipeline sheet in metres with each
  scene's scale fitted to its gold sheet;
- **`pipeline_v2`**, which **also uses the oracle scale**: the pipeline sheet with split objects merged and
  room/wall/floor/ceiling fragments removed, so `pipeline_v2` − `pipeline_oracle_scale` is the gain from that alone;
- **`gold_relations`**: the gold sheet plus precomputed centre-to-centre distances between the objects the question
  names and, for relative distance, the candidate closest to the target;
- **`gold_relations_v2`**: the same with **closest-point** (box-to-box) distances, which is how VSI-Bench's distance
  questions measure ("Measuring from the closest point of each object"), plus the two objects' distance for absolute
  distance;
- **`pipeline_relations`**, which **uses the oracle scale**: the `pipeline_v2` sheet plus the same closest-point
  relations computed from its own rows, so `pipeline_relations` against `gold_relations_v2` shows how much of the
  relations' gain survives with the pipeline's content.

The code is the format experiment's at commit `b381205`, plus `explore_sheets.py`.

| Scene | Questions | Video |
|---|---:|---|
| arkitscenes/42897528 | 7 | 1,428 frames at 30 fps, 47.6 s, 6.7 MB |
| scannet/scene0354_00 | 7 | 1,126 frames at 24 fps, 46.9 s, 4.9 MB |
| scannet/scene0356_00 | 7 | 1,360 frames at 24 fps, 56.7 s, 5.6 MB |
| scannet/scene0357_01 | 7 | 907 frames at 24 fps, 37.8 s, 5.8 MB |
| scannet/scene0591_01 | 12 | 1,724 frames at 24 fps, 71.8 s, 7.8 MB |

**Files on Google Drive**, in `MyDrive/mvp/`:

- `mvp-3d-vlm.zip`, the code, made in the repository with
  `git archive --format=zip -o mvp-3d-vlm.zip b381205 src experiments/2026-11-18-scene-sheet-formats requirements.txt pyproject.toml`
- `explore_ids.txt`, the 40 question ids, and `explore_sheets.py`, the extra conditions' sheets (neither is in
  `b381205`)
- the data, one of two ways (`DATA_SOURCE` in Paths):
  - `"huggingface"` (default): the scene-sheets zip (`SHEETS_ZIP`, the `scene_sheets_75` folder of
    `run_sheets.sh` zipped). VSI-Bench's `test.jsonl` and the 5 videos come from the Hugging Face dataset
    `nyu-visionx/VSI-Bench`: only those 5 files are read out of its per-dataset zips (range requests, about 32 MB, not
    the 1.8 and 2.9 GB zips). Each scene's `frame_indices.npy` and `meta.json` are taken from the sheets zip if it has
    them, else recomputed from the video as `mvp.reconstruction.extract.load_frames` does.
  - `"bundle"`: `ssf_explore_data.tar.gz`, made on the cluster with the command below.

Either way, two checks stop the notebook if the video's frames don't line up with the sheets: the 16 frames must be
the ones the sheets were made on (`adapter_report.json`), and every sheet's `first_seen` must be one of the
`<t seconds>` labels the model's processor puts on the video.

<details><summary>Making the data bundle on the cluster (alternative)</summary>

From the repository root. It holds VSI-Bench's `test.jsonl`, the sheets' `index.csv`, the 5 videos, each scene's
`frame_indices.npy` and `meta.json` from `data/vggt`, and each question's pipeline and gold sheet folders plus the
scene's `adapter_report.json`, for every question of the 5
scenes (no masks, frames or depth), with the sheets stored under `data/scene_sheets_75/`, where
the code looks by default. Sizes are printed before packing.

```bash
SHEETS=tasks/openvocab_adapter/out
LIST=$(mktemp)
python - "$SHEETS" > "$LIST" <<'EOF'
import csv, sys
sheets = sys.argv[1]
ids = {249, 566, 741, 742, 1142, 1267, 2961, 2962, 2966, 2972, 3115, 3237, 3245, 3248, 3433, 3647, 3707, 3769, 3770,
       3771, 3820, 4080, 4081, 4082, 4083, 4084, 4259, 4400, 4401, 4403, 4529, 4530, 4641, 4648, 4655, 4992, 5042, 5049,
       5056, 5058}
index = list(csv.DictReader(open(f"{sheets}/index.csv")))
scenes = {r["scene"] for r in index if int(r["id"]) in ids}
assert sum(int(r["id"]) in ids for r in index) == len(ids), "not every id is in index.csv"
paths = {"data/raw/VSI-Bench/test.jsonl", f"{sheets}/index.csv"}
for r in index:                                            # every sheet of these scenes (the oracle scale needs them)
    if r["scene"] not in scenes:
        continue
    scene = r["scene"]                                     # e.g. scannet/scene0354_00
    paths |= {f"data/raw/VSI-Bench/{scene}.mp4", f"data/vggt/{scene}/frame_indices.npy", f"data/vggt/{scene}/meta.json",
              f"{sheets}/{r['pipeline_sheet']}", f"{sheets}/{r['gold_sheet']}",
              f"{sheets}/{scene.replace('/', '__')}/subset-uniform16/scene/adapter_report.json"}
print("\n".join(sorted(paths)))
EOF
xargs -a "$LIST" ls -d > /dev/null && echo "all $(wc -l < "$LIST") paths exist"
xargs -a "$LIST" du -sh | sort -h | tail -n 12
xargs -a "$LIST" du -ch | tail -n 1
tar -czf ssf_explore_data.tar.gz --transform "s|^$SHEETS|data/scene_sheets_75|" -T "$LIST"
du -h ssf_explore_data.tar.gz
```
</details>

## Paths

In [ ]:
USE_DRIVE = True                     # False: put the files in /content yourself (Files pane) and set DRIVE_DIR to "/content"
DRIVE_DIR = "/content/drive/MyDrive/mvp"
CODE_ZIP = f"{DRIVE_DIR}/mvp-3d-vlm.zip"
IDS_FILE = f"{DRIVE_DIR}/explore_ids.txt"  # the questions
SHEETS_PY = f"{DRIVE_DIR}/explore_sheets.py"
SCENE_SCALE_PY = f"{DRIVE_DIR}/scene_scale.py"  # the scale from typical object sizes (not in b381205)
DATA_SOURCE = "huggingface"          # "huggingface": VSI-Bench from Hugging Face + SHEETS_ZIP | "bundle": DATA_TAR from the cluster
SHEETS_ZIP = f"{DRIVE_DIR}/scene_sheets_75.zip"  # the scene-sheets zip: set to its name on your Drive
DATA_TAR = f"{DRIVE_DIR}/ssf_explore_data.tar.gz"  # only for DATA_SOURCE = "bundle"
WORK = "/content/mvp-3d-vlm"         # the code and data are unpacked here
RESULTS_DIR = f"{DRIVE_DIR}/ssf_explore_results" if USE_DRIVE else "/content/ssf_explore_results"  # predictions (on Drive, they survive a disconnect and the runs resume)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# The settings of the format experiment's runs (its README, Runs) and the conditions: name -> (sheet source, format)
MODEL = "Qwen/Qwen3.5-2B"            # thinking off: the eval turns it off for Qwen3.5
EXP = f"{WORK}/experiments/2026-11-18-scene-sheet-formats"
IDS_PATH = f"{EXP}/{IDS_FILE.rsplit('/', 1)[-1]}"
SHEETS_ROOT = f"{WORK}/data/scene_sheets_75"
COMMON = ["--model", MODEL, "--nframes", "16", "--sampling", "uniform128", "--max-new-tokens", "2048", "--final-answer",
          "--ids", IDS_PATH]
ORACLE = "pipeline_oracle_scale"     # ORACLE UPPER BOUND, not a usable method: pipeline sheets in metres, scale fitted to gold
V2 = "pipeline_v2"                   # ALSO USES THE ORACLE SCALE: + split objects merged, + room/wall/floor/ceiling removed
RELATIONS = "gold_relations"         # the gold sheet + centre-to-centre relations between the objects the question names
GOLD_REL_V2 = "gold_relations_v2"    # the gold sheet + closest-point (box-to-box) relations, as VSI-Bench measures distance
PIPE_REL = "pipeline_relations"      # USES THE ORACLE SCALE: pipeline_v2 + the same closest-point relations
V2_INTERVALS = "pipeline_v2_intervals"  # USES THE ORACLE SCALE: pipeline_v2 with visible intervals, not first_seen / num_frames
SHUFFLED = "gold_csv_shuffled"       # the gold sheet with its rows in a random order (fixed per question), not by label
V2_PRIORS = "pipeline_v2_priors"     # pipeline_v2 with the scale from typical object sizes (scene_scale), NO gold
PIPE_REL_PRIORS = "pipeline_relations_priors"  # the same plus closest-point relations
ALL_CONDITIONS = {
    "A_video": None,
    "pipeline_csv": ("pipeline", "csv"),
    ORACLE: ("pipeline", "csv"),
    V2: ("pipeline", "csv"),
    V2_INTERVALS: ("pipeline", "csv"),
    V2_PRIORS: ("pipeline", "csv"),
    PIPE_REL: ("pipeline", "csv"),
    PIPE_REL_PRIORS: ("pipeline", "csv"),
    "pipeline_markdown_kv": ("pipeline", "markdown_kv"),
    "gold_csv": ("gold", "csv"),
    RELATIONS: ("gold", "csv"),
    SHUFFLED: ("gold", "csv"),
    GOLD_REL_V2: ("gold", "csv"),
    "gold_markdown_kv": ("gold", "markdown_kv"),
}
RUN = ['A_video', 'pipeline_csv', 'pipeline_oracle_scale', 'pipeline_v2', 'pipeline_relations', 'pipeline_markdown_kv', 'gold_csv', 'gold_relations', 'gold_relations_v2', 'gold_markdown_kv']  # the conditions this notebook runs and reports
CONDITIONS = {name: ALL_CONDITIONS[name] for name in RUN}
SHOW_SHEETS = 40                 # the input check prints the sheets of this many questions
# how the explore conditions differ: their sheets folder, the preamble in metres, the relations after the sheet
SHEET_ROOTS = {ORACLE: f"{WORK}/data/scene_sheets_75_oracle_scale", V2: f"{WORK}/data/scene_sheets_75_v2",
               PIPE_REL: f"{WORK}/data/scene_sheets_75_v2", V2_INTERVALS: f"{WORK}/data/scene_sheets_75_v2",
               V2_PRIORS: f"{WORK}/data/scene_sheets_75_v2_priors", PIPE_REL_PRIORS: f"{WORK}/data/scene_sheets_75_v2_priors"}
METRES = {ORACLE, V2, PIPE_REL, V2_INTERVALS}  # in metres with the ORACLE scale
PRIORS_SCALED = {V2_PRIORS, PIPE_REL_PRIORS}   # approximately in metres, the scale from typical object sizes
INTERVALS = {V2_INTERVALS}            # written with visible intervals (explore_sheets.render_intervals)
SHUFFLE = {SHUFFLED}                  # gold rows shuffled (explore_sheets.render_shuffled)
RELATION_MODE = {RELATIONS: "centre", GOLD_REL_V2: "closest", PIPE_REL: "closest", PIPE_REL_PRIORS: "closest"}

## Setup

In [ ]:
import os, shutil, subprocess, sys, zipfile

os.makedirs(WORK, exist_ok=True)
zipfile.ZipFile(CODE_ZIP).extractall(WORK)
shutil.copy(IDS_FILE, IDS_PATH)
shutil.copy(SHEETS_PY, f"{EXP}/explore_sheets.py")
shutil.copy(SCENE_SCALE_PY, f"{WORK}/src/mvp/benchmarks/scene_scale.py")

# requirements.txt pins torch 2.14.1+cu130; Colab's own torch is built for its driver, so it is kept unless this is False
KEEP_COLAB_TORCH = True
reqs = []
for line in open(f"{WORK}/requirements.txt").read().splitlines():
    line = line.split("#")[0].strip()
    if not line or (KEEP_COLAB_TORCH and (line.startswith("--extra-index-url") or line.split("==")[0] in ("torch", "torchvision"))):
        continue
    reqs.append(line)
# decord has no wheels for recent Python versions; eva-decord is a drop-in build imported under the same name
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[r for r in reqs if r != "decord"]], check=True)
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"]).returncode != 0:
    print("decord has no wheel for this Python; installing eva-decord instead")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eva-decord"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
sys.path.insert(0, f"{WORK}/src")
print("installed:", ", ".join(reqs))

In [ ]:
import torch

print("torch", torch.__version__, "CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > GPU (L4 or A100)"
name, cap = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
print(f"GPU: {name}, compute capability {cap[0]}.{cap[1]}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
if "T4" in name or cap < (8, 0):
    print("WARNING: this GPU has no native bfloat16 (T4 or older). The model runs in bfloat16, which is emulated here "
          "and several times slower; switch to an L4 or A100 if you can.")

## Data

Laid out as the code expects it: `data/raw/VSI-Bench/` (`test.jsonl`, `<dataset>/<scene>.mp4`),
`data/vggt/<dataset>/<scene>/` (`frame_indices.npy`, `meta.json`) and `data/scene_sheets_75/` (`index.csv`, the
sheet folders, `adapter_report.json`).

In [ ]:
import csv, io, json, shutil, tarfile, zipfile
from pathlib import Path

import numpy as np

VSI_REPO = "nyu-visionx/VSI-Bench"
RAW, VGGT, SHEETS = Path(WORK) / "data/raw/VSI-Bench", Path(WORK) / "data/vggt", Path(SHEETS_ROOT)
IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]


def save(data, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)


def export_frames(video, max_frames=128):
    """frame_indices and fps exactly as the VGGT export takes them (mvp.reconstruction.extract.load_frames): decord's
    frame count and average fps, and min(128, N) frames spaced by np.linspace(0, N - 1, dtype=int)."""
    from decord import VideoReader, cpu
    vr = VideoReader(str(video), ctx=cpu(0))
    n = len(vr)
    idx = np.linspace(0, n - 1, num=min(max_frames, n), dtype=int)
    return idx, n, float(vr.get_avg_fps())


if DATA_SOURCE == "bundle":
    with tarfile.open(DATA_TAR) as t:
        t.extractall(WORK)
    print("unpacked", DATA_TAR)
else:
    from huggingface_hub import HfFileSystem, hf_hub_download

    # 1. the sheets: index.csv, each question's pipeline and gold sheet folders, each scene's adapter_report.json
    with zipfile.ZipFile(SHEETS_ZIP) as z:
        names = z.namelist()
        index_name = min((n for n in names if n.endswith("index.csv")), key=len)  # the top-level one
        prefix = index_name[: -len("index.csv")]
        rows = {int(r["id"]): r for r in csv.DictReader(io.TextIOWrapper(z.open(index_name), encoding="utf-8"))}
        absent = [q for q in IDS if q not in rows]
        assert not absent, f"{SHEETS_ZIP}: questions {absent} are not in its index.csv"
        scenes = sorted({rows[q]["scene"] for q in IDS})
        # every sheet folder of these scenes, not only the questions': the oracle scale's alignment needs them all
        folders = {r[k] for r in rows.values() if r["scene"] in scenes for k in ("pipeline_sheet", "gold_sheet")}
        folders |= {f"{s.replace('/', '__')}/subset-uniform16/scene/adapter_report.json" for s in scenes}
        save(z.read(index_name), SHEETS / "index.csv")
        n_files = 0
        for name in names:
            rel = name[len(prefix):]
            if name.startswith(prefix) and not name.endswith("/") and any(rel == f or rel.startswith(f + "/") for f in folders):
                save(z.read(name), SHEETS / rel)
                n_files += 1
        print(f"sheets: index.csv and {n_files} files: every sheet of the {len(scenes)} scenes of the {len(IDS)} questions")
        # 2. frame_indices.npy and meta.json: from the zip if it has them for these scenes
        in_zip = {}
        for s in scenes:
            for f in ("frame_indices.npy", "meta.json"):
                hits = [n for n in names if n.endswith(f"/{s}/{f}") or n.endswith(f"/{s.replace('/', '__')}/{f}")]
                if hits:
                    in_zip[(s, f)] = min(hits, key=len)
        for s in scenes:
            if (s, "frame_indices.npy") in in_zip and (s, "meta.json") in in_zip:
                for f in ("frame_indices.npy", "meta.json"):
                    save(z.read(in_zip[(s, f)]), VGGT / s / f)
                print(f"{s}: frame_indices.npy and meta.json taken from the zip ({in_zip[(s, 'meta.json')]})")

    # 3. VSI-Bench's test.jsonl and the 5 videos, read out of the per-dataset zips without downloading them whole
    RAW.mkdir(parents=True, exist_ok=True)
    if not (RAW / "test.jsonl").exists():  # after a crash in the same runtime, what is here is not fetched again
        shutil.copy(hf_hub_download(VSI_REPO, "test.jsonl", repo_type="dataset"), RAW / "test.jsonl")
    fs = HfFileSystem()
    for dataset in sorted({s.split("/")[0] for s in scenes if not (RAW / f"{s}.mp4").exists()}):
        with fs.open(f"datasets/{VSI_REPO}/{dataset}.zip", "rb", block_size=2**20) as f, zipfile.ZipFile(f) as z:
            for s in [s for s in scenes if s.startswith(dataset + "/") and not (RAW / f"{s}.mp4").exists()]:
                save(z.read(f"{s}.mp4"), RAW / f"{s}.mp4")
                print(f"{s}.mp4: {(RAW / f'{s}.mp4').stat().st_size / 2**20:.1f} MB from {dataset}.zip")

    # 4. recompute frame_indices.npy and meta.json where the zip had none
    for s in scenes:
        if (VGGT / s / "meta.json").exists() and (VGGT / s / "frame_indices.npy").exists():
            continue
        idx, n, fps = export_frames(RAW / f"{s}.mp4")
        (VGGT / s).mkdir(parents=True, exist_ok=True)
        np.save(VGGT / s / "frame_indices.npy", idx.astype(np.int32))
        (VGGT / s / "meta.json").write_text(json.dumps({
            "video": f"data/raw/VSI-Bench/{s}.mp4", "num_video_frames": n, "video_fps": fps, "num_frames": len(idx),
            "recomputed": "explore.ipynb: frame_indices and video_fps as mvp.reconstruction.extract.load_frames "
                          "computes them; no VGGT outputs"}, indent=1))
        print(f"{s}: recomputed from the video: {n} frames, {fps} fps, {len(idx)} frame indices")

In [ ]:
# The video's "<t seconds>" labels come from the processor, from the fps the eval's video reader reports
# (qwen_vl_utils uses torchcodec, decord or torchvision, the first one installed). torchcodec measures the fps over the
# whole stream (frames / duration) and decord reads it from the header, and the two can differ in the last bits of the
# float (30.000000000000004 against 30.0), which flips a pair's label when its mean time is on a .x5 boundary. The sheets'
# timestamps (scene_sheet.frame_timestamps) use meta.json's video_fps, so it is set here to exactly what the eval's
# reader reports, read with the eval's own QwenVL.load_video; the export's value is kept as export_video_fps.
import inspect
import json
import sys

sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.models.qwen_vl import QwenVL
from qwen_vl_utils.vision_process import get_video_reader_backend

reader = QwenVL.__new__(QwenVL)  # load_video only needs QwenVL's pixel limits, not the model or the processor
limits = inspect.signature(QwenVL.__init__).parameters
reader.min_pixels, reader.max_pixels = limits["min_pixels"].default, limits["max_pixels"].default
backend = get_video_reader_backend()
index = ss.load_index(SHEETS)
changed = []
for s in sorted({index[q]["scene"] for q in IDS}):
    path = VGGT / s / "meta.json"
    meta = json.loads(path.read_text())
    _, video_meta, _ = reader.load_video(str(RAW / f"{s}.mp4"), nframes=ss.N_FRAMES)
    fps = float(video_meta["fps"])
    meta.setdefault("export_video_fps", meta["video_fps"])
    if meta["video_fps"] != fps or meta.get("video_reader_backend") != backend:
        if meta["video_fps"] != fps:
            changed.append(f"{s}: {meta['video_fps']!r} -> {fps!r}")
        meta.update(video_fps=fps, video_reader_backend=backend)
        path.write_text(json.dumps(meta, indent=1))
ss.frame_timestamps.cache_clear()  # timestamps computed before this cell used the old fps
print(f"video reader: {backend}; meta.json's video_fps changed to the reader's in {len(changed)} scenes"
      + "".join(f"\n  {c}" for c in changed))

In [ ]:
# Check 1: the 16 frames the eval picks (uniform128: 16 of the 128 frame_indices) are the frames the sheets were made on
# (adapter_report.json's source_video_frames). If not, the sheets describe other frames than the model sees.
import sys
sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.reconstruction.sa_sampling import uniform_subset

index = ss.load_index(SHEETS)
problems = []
for s in sorted({index[q]["scene"] for q in IDS}):
    idx = np.load(VGGT / s / "frame_indices.npy")
    report = json.loads((SHEETS / s.replace("/", "__") / "subset-uniform16" / "scene" / "adapter_report.json").read_text())
    picked = idx[uniform_subset(len(idx), 16)].tolist()
    fps = json.loads((VGGT / s / "meta.json").read_text())["video_fps"]
    print(f"{s}: {len(idx)} frame indices, fps {fps}; the eval's 16 frames {picked}")
    if report["selected_frames"] != uniform_subset(128, 16):
        problems.append(f"{s}: the sheets were made on frames {report['selected_frames']} of the 128, not uniform16")
    if picked != report["source_video_frames"]:
        problems.append(f"{s}: the eval would show video frames {picked}, but the sheets were made on "
                        f"{report['source_video_frames']}")
if problems:
    raise RuntimeError("The video frames don't match the sheets, so the sheets would describe other frames than the "
                       "model sees and first_seen would point at the wrong moments:\n  " + "\n  ".join(problems) +
                       "\nThe videos or the recomputed frame_indices.npy differ from the cluster's. Use the cluster's "
                       "data (DATA_SOURCE = \"bundle\") instead.")
print("OK: every scene's 16 frames are the ones its sheets were made on")

## Explore sheets: oracle scale, pipeline_v2, gold relations

**`pipeline_oracle_scale` is an oracle upper bound, not a usable method.** Its pipeline sheets are put in metres with
each scene's scale fitted to that scene's **gold** sheet, which no real system has. Everything else is the
`pipeline_csv` sheet; only the numbers are in metres and the preamble says so.

**`pipeline_v2` also uses that oracle scale.** On top of it, split objects are merged (rows of one class never seen in
the same frame, with centres within `explore_sheets.MERGE_THRESHOLD` metres or overlapping boxes) and room / wall /
floor / ceiling fragments are removed, except where a question other than counting or room size names them (route
landmarks, the ceiling). `pipeline_v2` − `pipeline_oracle_scale` is the gain from merging and filtering alone; the
threshold and the sheets' counts and sizes against gold are in `sheet_quality.py --v2` (section 6). Room-size
questions name only `room`, so their `pipeline_v2` sheet is empty and shows the empty-sheet line.

**`gold_relations`** is the gold sheet with, after it, centre-to-centre distances in metres between the objects the
question names and, for relative distance, each candidate's distance to the target and the closest one.
**`gold_relations_v2`** gives closest-point (box-to-box) distances instead, as VSI-Bench's distance questions measure
them, and for absolute distance the two objects' distance. **`pipeline_relations`** is the `pipeline_v2` sheet (oracle
scale) with the same closest-point relations computed from its own rows. With gold content these relations state the
answer to most distance questions; they show whether the model uses relations it is given, not spatial reasoning.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, f"{WORK}/src")
sys.path.insert(0, EXP)
import numpy as np
import explore_sheets as es
from mvp.benchmarks import scene_sheet as ss

index = ss.load_index(Path(SHEETS_ROOT))
IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
scenes = sorted({index[q]["scene"] for q in IDS})
SCALES = es.oracle_scales(SHEETS_ROOT, scenes)  # ORACLE: fitted to each scene's gold sheet
assert set(SCALES) == set(scenes), f"no oracle scale for {sorted(set(scenes) - set(SCALES))}"
display(pd.DataFrame(SCALES).T.rename_axis("scene"))
for name, variant in [(ORACLE, "oracle"), (V2, "v2")]:
    n = es.write_sheets(SHEETS_ROOT, SHEET_ROOTS[name], SCALES, variant)
    print(f"{name}: {n} pipeline sheets rewritten in {SHEET_ROOTS[name]}")
METRES_PREAMBLE = es.oracle_preamble(ss.PREAMBLES["pipeline"])
INTERVALS_PREAMBLE = es.intervals_preamble(METRES_PREAMBLE)  # pipeline_v2_intervals: visible instead of first_seen
PRIORS_PREAMBLE = es.priors_preamble(ss.PREAMBLES["pipeline"])  # pipeline_v2_priors: "approximately in metres"

# pipeline_v2_priors: the same merging and filtering, with the scale from typical object sizes instead of the oracle's
from mvp.benchmarks import scene_scale
n = es.write_sheets(SHEETS_ROOT, SHEET_ROOTS[V2_PRIORS], es.priors_scale(SHEETS_ROOT), "v2")
print(f"{V2_PRIORS}: {n} pipeline sheets rewritten in {SHEET_ROOTS[V2_PRIORS]}")
q_scale = {q: scene_scale.question_scale(q, SHEETS_ROOT) for q in IDS}
display(pd.DataFrame([{"scene": s, "oracle": round(SCALES[s]["scale"], 3), "oracle method": SCALES[s]["method"],
                       "priors": round(np.median([q_scale[q]["k"] for q in IDS if index[q]["scene"] == s]), 3)}
                      for s in scenes]).set_index("scene"))
print(f"priors scale: {sum(v['fallback'] for v in q_scale.values())} questions fall back to scene_scale.FALLBACK, "
      f"{sum(bool(v['left_out']) for v in q_scale.values())} object-size questions leave their class out")
print("\npreamble of", " and ".join(sorted(METRES)) + ":", METRES_PREAMBLE)

# pipeline_v2 against pipeline_oracle_scale, question by question: rows, and empty sheets
rows = lambda root, q: len(es.read_csv(Path(root) / index[q]["pipeline_sheet"] / "objects.csv")[0])
v2_rows = pd.DataFrame([{"id": q, "type": index[q]["question_type"], "oracle rows": rows(SHEET_ROOTS[ORACLE], q),
                         "v2 rows": rows(SHEET_ROOTS[V2], q)} for q in IDS]).set_index("id")
print(f"\npipeline_v2: {v2_rows['v2 rows'].sum()} rows against {v2_rows['oracle rows'].sum()}; empty sheets "
      f"{(v2_rows['v2 rows'] == 0).sum()} against {(v2_rows['oracle rows'] == 0).sum()} "
      f"({', '.join(sorted(set(v2_rows[v2_rows['v2 rows'] == 0]['type'])))})")

## Input check

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, f"{WORK}/src")
from decord import VideoReader
from mvp.benchmarks import scene_sheet as ss
from mvp.benchmarks.vsibench import load_vsibench
from mvp.paths import VGGT_DIR, VSI_DIR

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
index = ss.load_index(Path(SHEETS_ROOT))
vsi = load_vsibench("full").set_index("id")
need = [VSI_DIR / "test.jsonl", Path(SHEETS_ROOT) / "index.csv"]
for q in IDS:
    row = index[q]
    pipe = Path(SHEETS_ROOT) / row["pipeline_sheet"]
    need += [Path(vsi.loc[q, "video_path"]), VGGT_DIR / row["scene"] / "frame_indices.npy",
             VGGT_DIR / row["scene"] / "meta.json", pipe / "objects.csv", pipe / "question.json",
             pipe.parents[1] / "scene" / "adapter_report.json", Path(SHEETS_ROOT) / row["gold_sheet"] / "objects.csv"]
missing = sorted({str(p) for p in need if not p.exists()})
assert not missing, "missing files:\n" + "\n".join(missing)
assert all(vsi.loc[q, "question"] == index[q]["question"] for q in IDS), "index.csv and test.jsonl disagree on a question"
print(f"all {len(set(map(str, need)))} files present for {len(IDS)} questions")

for scene in sorted({index[q]["scene"] for q in IDS}):
    video = Path(vsi[vsi["scene_name"] == scene.split("/")[1]]["video_path"].iloc[0])
    vr = VideoReader(str(video))
    fps = json.loads((VGGT_DIR / scene / "meta.json").read_text())["video_fps"]
    print(f"{scene}: {len(vr)} frames, {fps:.2f} fps, {len(vr) / fps:.0f} s, "
          f"{sum(index[q]['scene'] == scene for q in IDS)} questions")

In [ ]:
# Check 2: the prompts as the eval builds them (--print-prompt: the processor and the videos, not the model's weights),
# checked by the format experiment's cluster_checks.check_prompts: the video's "<t seconds>" labels are the sheet's
# timestamps, every first_seen in the sheet is one of those labels, the sheet is in the prompt, and the whole prompt is
# build_sheet_prompt's. Pipeline sheets carry first_seen; gold sheets share the same video labels.
import os, subprocess
sys.path.insert(0, EXP)
import cluster_checks

cmd = [sys.executable, "-m", "mvp.eval.vsibench", *COMMON, "--sheet-source", "pipeline", "--sheet-format", "csv",
       "--sheets-root", SHEETS_ROOT, "--print-prompt", str(len(IDS))]
printed = subprocess.run(cmd, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"}, capture_output=True, text=True)
assert printed.returncode == 0, printed.stdout[-3000:] + printed.stderr[-3000:]
problems = cluster_checks.check_prompts(printed.stdout, Path(SHEETS_ROOT), VGGT_DIR)
if problems:
    raise RuntimeError("The video's timestamp labels don't match the sheets, so first_seen would name moments the "
                       "model's video doesn't show:\n  " + "\n  ".join(problems) + "\nThe fps cell above sets "
                       "meta.json's video_fps to the eval reader's; if this still fails, the reader or the indices differ.")
print(f"OK: {printed.stdout.count('===== question')} prompts; every first_seen is one of the video's labels")

In [ ]:
# Each question's sheet in the sheet conditions, rendered by the eval's own function (scene_sheet.render_sheet),
# exactly as it goes between "These are frames of a video." and the question.
# The explore conditions go through explore_sheets.render, which the run's entry point (explore_sheets.py) matches.
def render(q, name):
    source, fmt = CONDITIONS[name]
    return es.render(q, source, fmt, SHEET_ROOTS.get(name, SHEETS_ROOT), VGGT_DIR,
                     preamble=sheet_preamble(name), relations_mode=RELATION_MODE.get(name),
                     intervals=name in INTERVALS, shuffle=name in SHUFFLE)


def sheet_preamble(name):
    return (INTERVALS_PREAMBLE if name in INTERVALS else PRIORS_PREAMBLE if name in PRIORS_SCALED
            else METRES_PREAMBLE if name in METRES else None)


def label(name):
    return {ORACLE: "  (ORACLE: scale fitted to gold)", V2: "  (uses the ORACLE scale)",
            PIPE_REL: "  (uses the ORACLE scale)", V2_INTERVALS: "  (uses the ORACLE scale)",
            V2_PRIORS: "  (PRIORS scale: typical object sizes, no gold)",
            PIPE_REL_PRIORS: "  (PRIORS scale: typical object sizes, no gold)"}.get(name, "")

for q in IDS[:SHOW_SHEETS]:
    row = index[q]
    print("=" * 100)
    print(f"question {q} ({row['question_type']}, {row['scene']}): {row['question']}")
    if row["options"]:
        print("options:", row["options"])
    print("ground truth:", row["ground_truth"])
    for name, sheet in CONDITIONS.items():
        if sheet:
            print(f"\n--- {name}{label(name)} ---")
            print(render(q, name))

In [ ]:
# The whole text input for one question in every condition (build_sheet_prompt: chat template, the video's
# "<t seconds>" labels with one <|video_pad|> per frame pair, the sheet, the question). Change Q to see another.
Q = IDS[0]
for name, sheet in CONDITIONS.items():
    print(f"===== {name}{label(name)} =====")
    prompt = ss.build_sheet_prompt(Q, *(sheet or (None, None)), Path(SHEET_ROOTS.get(name, SHEETS_ROOT)), VGGT_DIR)
    if sheet:  # the sheet as the explore condition renders it
        plain = ss.render_sheet(Q, *sheet, Path(SHEET_ROOTS.get(name, SHEETS_ROOT)), vggt_root=VGGT_DIR)
        prompt = prompt.replace(plain, render(Q, name))
    print(prompt)

## Runs

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]


def answered(path):
    """The question ids a prediction file already answers. A crash can leave a half-written last line, which would stop
    the eval's own resume (it reads every line as JSON): unreadable lines are dropped from the file first."""
    path = Path(path)
    if not path.exists():
        return set()
    good = []
    for line in path.read_text().splitlines():
        try:
            good.append(json.loads(line))
        except json.JSONDecodeError:
            pass
    if len(good) != len(path.read_text().splitlines()):
        print(f"{path.name}: dropped {len(path.read_text().splitlines()) - len(good)} unreadable line(s) left by a crash")
        path.write_text("".join(json.dumps(r) + "\n" for r in good))
    return {r["id"] for r in good}


os.makedirs(RESULTS_DIR, exist_ok=True)
for name, sheet in CONDITIONS.items():
    # resume: a condition with every question answered is skipped without loading the model; a partial one continues
    # (the eval skips the questions already in its --out file)
    done = answered(f"{RESULTS_DIR}/{name}.jsonl") & set(IDS)
    if len(done) == len(IDS):
        print(f"===== {name}: all {len(IDS)} questions already answered, skipped", flush=True)
        continue
    if done:
        print(f"===== {name}: resuming, {len(done)} of {len(IDS)} answered", flush=True)
    # the explore conditions run the same eval through explore_sheets.py, which only swaps the pipeline preamble
    # (EXPLORE_PREAMBLE), writes intervals (EXPLORE_INTERVALS), shuffles gold rows (EXPLORE_SHUFFLE) and appends the
    # relations (EXPLORE_RELATIONS)
    explore = name in SHEET_ROOTS or name in RELATION_MODE or name in SHUFFLE
    entry = [f"{EXP}/explore_sheets.py"] if explore else ["-m", "mvp.eval.vsibench"]
    preamble = (INTERVALS_PREAMBLE if name in INTERVALS else PRIORS_PREAMBLE if name in PRIORS_SCALED
                else METRES_PREAMBLE if name in METRES else "")
    env = {**os.environ, "PYTHONPATH": f"{WORK}/src", "EXPLORE_PREAMBLE": preamble,
           "EXPLORE_RELATIONS": RELATION_MODE.get(name, ""), "EXPLORE_INTERVALS": "1" if name in INTERVALS else "",
           "EXPLORE_SHUFFLE": "1" if name in SHUFFLE else ""}
    cmd = [sys.executable, *entry, *COMMON, "--out", f"{RESULTS_DIR}/{name}.jsonl"]
    if sheet:
        cmd += ["--sheet-source", sheet[0], "--sheet-format", sheet[1], "--sheets-root", SHEET_ROOTS.get(name, SHEETS_ROOT)]
    print(f"===== {name}: {' '.join(cmd[1:])}", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    assert proc.wait() == 0, f"{name} failed"
    print(f"===== {name} done in {time.time() - t0:.0f} s\n", flush=True)

## Per-question table

In [ ]:
import csv, sys
from pathlib import Path
from collections import Counter

import pandas as pd

sys.path.insert(0, f"{WORK}/src")
from mvp.benchmarks import scene_sheet as ss
from mvp.benchmarks.vsibench import MCA_QUESTION_TYPES, final_letter, final_number, summarize, task_of

IDS = [int(x) for x in Path(IDS_PATH).read_text().split()]
index = ss.load_index(Path(SHEETS_ROOT))
runs = {}
for name in CONDITIONS:
    path = Path(RESULTS_DIR) / f"{name}.jsonl"
    if path.exists():
        runs[name] = pd.read_json(path, lines=True).drop_duplicates("id").set_index("id")
missing = [n for n in CONDITIONS if n not in runs]
print("conditions with predictions:", ", ".join(runs) or "none", "| missing:", ", ".join(missing) or "none")
for name, run in runs.items():
    if set(run.index) != set(IDS):
        print(f"WARNING {name}: {len(set(IDS) - set(run.index))} questions without an answer")


def objects(folder):
    """The sheet's rows as 'chair x3, table x1', or '(empty)'."""
    with open(Path(SHEETS_ROOT) / folder / "objects.csv", newline="") as f:
        counts = Counter(r["concept"] for r in csv.DictReader(f))
    return ", ".join(f"{c} x{n}" for c, n in sorted(counts.items())) or "(empty)"


def answer(qtype, prediction):
    """The answer the final-answer rule reads (score_final): a letter, a number, or None."""
    if qtype in MCA_QUESTION_TYPES:
        return final_letter(str(prediction))
    return final_number(str(prediction))


base = pd.DataFrame([{"id": q, "scene": index[q]["scene"], "type": index[q]["question_type"],
                      "task": task_of(index[q]["question_type"]), "question": index[q]["question"],
                      "truth": index[q]["ground_truth"], "pipeline sheet": objects(index[q]["pipeline_sheet"]),
                      "gold sheet": objects(index[q]["gold_sheet"])} for q in IDS]).set_index("id")
base["pipeline empty"] = base["pipeline sheet"] == "(empty)"
answers, scores = pd.DataFrame(index=base.index), pd.DataFrame(index=base.index)
for name, run in runs.items():
    answers[name] = [answer(base.loc[q, "type"], run.loc[q, "prediction"]) if q in run.index else None for q in base.index]
    scores[name] = [float(run.loc[q, "score"]) if q in run.index else float("nan") for q in base.index]


def cell(q, name):
    a, s = answers.loc[q, name], scores.loc[q, name]
    if pd.isna(s):
        return "—"
    if base.loc[q, "type"] in MCA_QUESTION_TYPES:
        return f"{a} {'✓' if s == 1 else '✗'}"
    return f"{'?' if a is None else f'{a:g}'} ({s:.2f})"


table = base[["scene", "type", "truth"]].copy()
for name in runs:
    table[name] = [cell(q, name) for q in base.index]
table[["pipeline sheet", "gold sheet"]] = base[["pipeline sheet", "gold sheet"]]
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
display(table)

Full replies, for any question: `runs['pipeline_csv'].loc[QUESTION_ID, 'prediction']`.

## Summary

`pipeline_oracle_scale` is an oracle upper bound (scale fitted to gold), not a usable method; `pipeline_v2` and `pipeline_relations` use the same oracle scale; `pipeline_v2_priors` and `pipeline_relations_priors` use the scale from typical object sizes, with no gold.

In [ ]:
# Score x100 per condition: per task (relative direction's three types together) and VSI-Bench's overall (the mean of
# the 8 tasks), then per question type, then split by whether the pipeline sheet is empty.
long = pd.DataFrame([{"condition": name, "id": q, "question_type": base.loc[q, "type"], "score": scores.loc[q, name]}
                     for name in runs for q in base.index]).dropna()
by_task = pd.DataFrame({name: summarize(g) for name, g in long.groupby("condition", sort=False)}).T
by_task = by_task[[c for c in by_task.columns if c != "overall"] + ["overall"]].round(1)
display(by_task)

by_type = (100 * long.pivot_table(index="question_type", columns="condition", values="score", aggfunc="mean",
                                  sort=False)).round(1)[list(runs)]
by_type.insert(0, "n", base["type"].value_counts())
display(by_type)

long["pipeline sheet"] = ["empty" if base.loc[q, "pipeline empty"] else "not empty" for q in long["id"]]
by_empty = (100 * long.pivot_table(index="pipeline sheet", columns="condition", values="score", aggfunc="mean",
                                   sort=False)).round(1)[list(runs)]
by_empty.insert(0, "n", base["pipeline empty"].map({True: "empty", False: "not empty"}).value_counts())
display(by_empty)

## Bootstrap intervals

Which differences from video only are real: 95% intervals from 10,000 paired, per-question-type resamples of the questions. "not clear" means the interval for the difference includes 0.

In [ ]:
# Bootstrap 95% intervals for VSI-Bench's overall score (the mean of the 8 task scores, relative direction's three
# types averaged into one task, as summarize computes it) and for each condition's difference from A_video. Questions
# are resampled with replacement within each question type, so every type keeps its number of questions, and every
# condition gets the same resample, so the differences are paired. Only questions every condition answered are used.
import numpy as np

B = 10_000
rng = np.random.default_rng(0)
conds = list(runs)
common = [q for q in base.index if all(pd.notna(scores.loc[q, n]) for n in conds)]
S = scores.loc[common, conds].to_numpy(float)
qtypes = sorted(set(base.loc[common, "type"]))
qtype_of = np.array([qtypes.index(t) for t in base.loc[common, "type"]])
tasks = sorted({task_of(t) for t in qtypes})
# task score = the mean of its question types' means; overall = the mean of the task scores
weights = np.array([[(task_of(t) == k) / sum(task_of(u) == k for u in qtypes) for t in qtypes] for k in tasks])
members = [np.flatnonzero(qtype_of == i) for i in range(len(qtypes))]


def overall(sample):
    """Overall x100 per condition for questions `sample` (indices into common, grouped as members)."""
    means = np.stack([S[sample[i]].mean(0) for i in range(len(qtypes))])  # qtype x condition
    return 100 * (weights @ means).mean(0)


point = overall(members)
check = summarize(pd.DataFrame({"question_type": base.loc[common, "type"].to_numpy(), "score": S[:, 0]}))["overall"]
assert abs(point[0] - check) < 1e-9, (point[0], check)  # the same overall as summarize
boot = np.stack([overall([rng.choice(m, size=len(m)) for m in members]) for _ in range(B)])
ci = pd.DataFrame({"overall": point, "95% low": np.percentile(boot, 2.5, axis=0),
                   "95% high": np.percentile(boot, 97.5, axis=0)}, index=conds)
if "A_video" in conds:
    a = conds.index("A_video")
    diff = boot - boot[:, [a]]
    ci["minus A"] = point - point[a]
    ci["minus A, 95% low"] = np.percentile(diff, 2.5, axis=0)
    ci["minus A, 95% high"] = np.percentile(diff, 97.5, axis=0)
    ci["differs from A"] = np.where(ci.index == "A_video", "",
                                    np.where(ci["minus A, 95% low"] > 0, "higher",
                                             np.where(ci["minus A, 95% high"] < 0, "lower", "not clear")))
print(f"{len(common)} questions answered in every condition; {B:,} stratified paired resamples")
display(ci.round(1))

## What the real scale costs: priors against oracle

In [ ]:
# What the real scale costs: each sheet with the scale from typical object sizes (no gold) against the same sheet with
# the oracle scale (fitted to gold). Overall from the bootstrap above (the same paired resamples), and the three tasks
# asked in metres.
SCALE_PAIRS = [("pipeline_v2", "pipeline_v2_priors"), ("pipeline_relations", "pipeline_relations_priors")]
METRIC_TASKS = ["object_abs_distance", "object_size_estimation", "room_size_estimation"]
cost = []
for oracle, priors in SCALE_PAIRS:
    if oracle not in conds or priors not in conds:
        print(f"{oracle} or {priors} has no predictions: no comparison")
        continue
    i, j = conds.index(oracle), conds.index(priors)
    d = boot[:, j] - boot[:, i]
    cost.append({"sheet": oracle, "oracle scale": round(point[i], 1), "priors scale": round(point[j], 1),
                 "priors minus oracle": round(point[j] - point[i], 1),
                 "95% interval": f"[{np.percentile(d, 2.5):.1f}, {np.percentile(d, 97.5):.1f}]",
                 **{f"{t}: oracle / priors": f"{by_task.loc[oracle, t]:.1f} / {by_task.loc[priors, t]:.1f}"
                    for t in METRIC_TASKS if t in by_task.columns}})
display(pd.DataFrame(cost).set_index("sheet") if cost else "no pair has both scales")

## Appearance order: first_seen against intervals, sorted against shuffled rows

In [ ]:
# Appearance order, two versions side by side: pipeline_v2 with first_seen against pipeline_v2_intervals with visible
# intervals, and gold_csv (rows sorted by label) against gold_csv_shuffled (rows in a random order). Score x100 on the
# appearance-order questions every condition answered, with bootstrap 95% intervals: questions resampled with
# replacement, the same resample for both versions, so the difference is paired. Few questions: read the intervals.
PAIRS = [("pipeline_v2", "pipeline_v2_intervals", "first_seen", "visible intervals"),
         ("gold_csv", "gold_csv_shuffled", "rows sorted by label", "rows shuffled")]
app = [i for i, q in enumerate(common) if base.loc[q, "type"] == "obj_appearance_order"]
draws = np.random.default_rng(1).integers(0, len(app), size=(B, len(app)))


def interval(values):
    return f"[{np.percentile(values, 2.5):.1f}, {np.percentile(values, 97.5):.1f}]"


side = []
for a, b, la, lb in PAIRS:
    if a not in conds or b not in conds:
        print(f"{a} or {b} has no predictions: no comparison")
        continue
    sa, sb = S[app, conds.index(a)], S[app, conds.index(b)]
    ba, bb = 100 * sa[draws].mean(1), 100 * sb[draws].mean(1)
    side.append({"pair": f"{a} | {b}", "version 1": la, "score 1": round(100 * sa.mean(), 1), "95% 1": interval(ba),
                 "version 2": lb, "score 2": round(100 * sb.mean(), 1), "95% 2": interval(bb),
                 "2 minus 1": round(100 * (sb.mean() - sa.mean()), 1), "95% difference": interval(bb - ba)})
if "A_video" in conds:
    va = 100 * S[app, conds.index("A_video")][draws].mean(1)
    print(f"A_video on the same {len(app)} appearance-order questions: "
          f"{100 * S[app, conds.index('A_video')].mean():.1f} {interval(va)}")
display(pd.DataFrame(side).set_index("pair") if side else "no pair has both versions")

## Appearance order: does the model read row order as time?

In [ ]:
# Does the model read the sheet's row order as appearance order? Gold sheets have no time information, but their rows
# are sorted by label; pipeline rows by first_seen. For each appearance-order question: the correct order, the order
# each sheet lists the named objects in, and each condition's answer. Agreement is the share of object pairs that the
# chosen option orders as the sheet's rows do (1: the same order, 0.5: unrelated on average). If gold_csv's answers
# agree with the gold row order more than A_video's do, the rows are being read as time; gold_csv_shuffled's answers
# should then follow its own shuffled rows instead.
import itertools
import re

sys.path.insert(0, EXP)
import explore_sheets as es


def named(q):
    return [x.strip() for x in re.search(r"following categories in the video: (.*)\?",
                                         index[q]["question"]).group(1).split(",")]


def options(q):
    return {o.split(". ", 1)[0]: [x.strip() for x in o.split(". ", 1)[1].split(",")]
            for o in index[q]["options"].split(" | ")}


def first_order(labels, names):
    """The named classes in the order they first occur in a sheet's rows."""
    return list(dict.fromkeys(l for l in labels if l in names))


def gold_rows(q, shuffled=False):
    objects = ss.load_objects(q, "gold", Path(SHEETS_ROOT))
    if shuffled:
        es.random.Random(f"{es.SHUFFLE_SEED}:{q}").shuffle(objects)  # as explore_sheets.render_shuffled
    return first_order([o["label"] for o in objects], named(q))


def pipeline_rows(q):
    rows = es.read_csv(Path(SHEETS_ROOT) / index[q]["pipeline_sheet"] / "objects.csv")[0]
    first = lambda r: min([t for t in range(16) if r[f"frame_{t:03d}"] == "1"], default=16)
    return first_order([r["concept"] for r in sorted(rows, key=lambda r: (first(r), r["concept"], int(r["track_id"])))],
                       named(q))


def agreement(order, ref):
    pairs = [(a, b) for a, b in itertools.combinations([x for x in order if x in ref], 2)]
    return float(np.mean([ref.index(a) < ref.index(b) for a, b in pairs])) if pairs else np.nan


qs_app = [q for q in base.index if base.loc[q, "type"] == "obj_appearance_order"]
detail, agree = [], []
for q in qs_app:
    opts, truth = options(q), index[q]["ground_truth"]
    refs = {"gold rows": gold_rows(q), "shuffled gold rows": gold_rows(q, True), "pipeline rows": pipeline_rows(q)}
    rec = {"id": q, "correct": f"{truth}: {', '.join(opts[truth])}", **{k: ", ".join(v) for k, v in refs.items()}}
    for name in runs:
        letter = answers.loc[q, name]
        rec[name] = f"{letter} {'✓' if letter == truth else '✗'}" if letter in opts else "?"
        if letter in opts:
            agree.append({"condition": name, "with correct": agreement(opts[letter], opts[truth]),
                          **{f"with {k}": agreement(opts[letter], v) for k, v in refs.items()}})
    detail.append(rec)
display(pd.DataFrame(detail).set_index("id"))
summary_order = pd.DataFrame(agree).groupby("condition", sort=False).mean().round(2)
summary_order.insert(0, "answers", pd.DataFrame(agree).groupby("condition", sort=False).size())
display(summary_order)

## Changed answers

In [ ]:
# For every sheet condition: the questions whose answer differs from A (video only), and whether the score went up
# (helped), down (hurt) or stayed the same.
def same(a, b):
    if a is None or b is None or isinstance(a, str) or isinstance(b, str):
        return a == b
    return abs(float(a) - float(b)) < 1e-9


changes, rows = [], []
if "A_video" in runs:
    for name in [n for n in runs if n != "A_video"]:
        for q in base.index:
            a, b = answers.loc[q, "A_video"], answers.loc[q, name]
            if same(a, b) or pd.isna(scores.loc[q, name]) or pd.isna(scores.loc[q, "A_video"]):
                continue
            d = scores.loc[q, name] - scores.loc[q, "A_video"]
            effect = "helped" if d > 0 else "hurt" if d < 0 else "same score"
            changes.append({"condition": name, "id": q, "type": base.loc[q, "type"], "truth": base.loc[q, "truth"],
                            "A answer": a, "answer": b, "A score": round(scores.loc[q, "A_video"], 2),
                            "score": round(scores.loc[q, name], 2), "effect": effect,
                            "pipeline empty": base.loc[q, "pipeline empty"]})
    changes = pd.DataFrame(changes, columns=["condition", "id", "type", "truth", "A answer", "answer", "A score", "score",
                                             "effect", "pipeline empty"])
    counts = (changes.groupby(["condition", "effect"]).size().unstack(fill_value=0)
              .reindex(index=[n for n in runs if n != "A_video"], columns=["helped", "hurt", "same score"], fill_value=0))
    counts["changed"] = counts.sum(axis=1)
    counts["of"] = [int(scores[n].notna().sum()) for n in counts.index]
    display(counts)
    display(changes.sort_values(["condition", "effect", "type", "id"]).set_index(["condition", "id"]))
else:
    print("no A_video predictions: nothing to compare with")

## Diagnosis: markdown_kv against csv

In [ ]:
# Is pipeline_markdown_kv's low score a bug? Parse rate per condition (the final-answer rule, as the scores use), then
# the raw replies for questions where markdown_kv scored lower than csv.
from mvp.benchmarks.vsibench import parsed

rate = pd.DataFrame({name: {"parsed %": 100 * sum(parsed(r, r["prediction"], final=True) for _, r in run.iterrows()) / len(run),
                            "first word = final answer %": 100 * (run["score"] == run["score_first"]).mean(),
                            "median reply chars": run["prediction"].astype(str).str.len().median(),
                            "max reply chars": run["prediction"].astype(str).str.len().max(),
                            "score": 100 * run["score"].mean()} for name, run in runs.items()}).T.round(1)
display(rate)

kv, csv_ = runs.get("pipeline_markdown_kv"), runs.get("pipeline_csv")
worse = [] if kv is None or csv_ is None else [
    q for q in base.index if q in kv.index and q in csv_.index and kv.loc[q, "score"] < csv_.loc[q, "score"]]
worse.sort(key=lambda q: (base.loc[q, "type"] not in MCA_QUESTION_TYPES, kv.loc[q, "score"] - csv_.loc[q, "score"]))
if kv is None or csv_ is None:
    print("pipeline_markdown_kv or pipeline_csv was not run: no comparison")
else:
    print(f"{len(worse)} questions where markdown_kv scored lower than csv; the first 5 (multiple choice first, then "
          "the largest drops):")
for q in worse[:5]:
    print("=" * 100)
    print(f"question {q} ({base.loc[q, 'type']}): {base.loc[q, 'question']}\nground truth: {base.loc[q, 'truth']}")
    for name, run in (("csv", csv_), ("markdown_kv", kv)):
        print(f"--- {name}: score {run.loc[q, 'score']:.2f}, read as {answer(base.loc[q, 'type'], run.loc[q, 'prediction'])!r}, "
              f"first-word score {run.loc[q, 'score_first']:.2f}")
        print(repr(run.loc[q, "prediction"]))

## Save

In [ ]:
# The tables next to the predictions (on Drive when USE_DRIVE).
table.to_csv(f"{RESULTS_DIR}/per_question.csv")
by_task.to_csv(f"{RESULTS_DIR}/by_task.csv")
by_type.to_csv(f"{RESULTS_DIR}/by_question_type.csv")
ci.to_csv(f"{RESULTS_DIR}/overall_bootstrap.csv")
if len(changes):
    changes.to_csv(f"{RESULTS_DIR}/changed_answers.csv", index=False)
print("saved to", RESULTS_DIR)